In [ ]:
from google.colab import files
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

uploaded = files.upload()
filename = next(
    name for name in uploaded
    if name.lower().endswith(".csv")
)
df = pd.read_csv(io.BytesIO(uploaded[filename]))
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="raise")

features = [
    "temperature_mean",
    "temperature_std",
    "pressure_mean",
    "process_time",
    "alignment_error",
    "surface_particle_count",
    "runs_since_maintenance",
]

assert set(features + ["defect_flag"]).issubset(df.columns)
assert df["defect_flag"].notna().all(), "Missing target labels"
assert df["defect_flag"].isin([0, 1]).all(), "Invalid target labels"

df[features] = df[features].apply(pd.to_numeric, errors="raise")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Defect rate: {df['defect_flag'].mean():.2%}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique_values": df.nunique(),
}))

display(df[features].describe().round(3))
display(df.head())

In [ ]:
baseline = df["defect_flag"].mean()
bin_tables = []

fig, axes = plt.subplots(3, 3, figsize=(17, 12))
axes = axes.ravel()

for ax, feature in zip(axes, features):
    temp = df[[feature, "defect_flag"]].copy()
    temp = temp.replace([np.inf, -np.inf], np.nan).dropna()

    if temp[feature].nunique() < 2:
        ax.set_title(f"{feature}: insufficient variation")
        continue

    temp["bin"] = pd.qcut(temp[feature], q=10, duplicates="drop")

    summary = (
        temp.groupby("bin", observed=True)
        .agg(
            parameter_median=(feature, "median"),
            sample_count=("defect_flag", "size"),
            defect_count=("defect_flag", "sum"),
            defect_rate=("defect_flag", "mean"),
        )
        .reset_index()
    )

    summary.insert(0, "feature", feature)
    bin_tables.append(summary)

    ax.plot(
        summary["parameter_median"],
        summary["defect_rate"] * 100,
        marker="o",
        color="#E45756",
    )
    ax.axhline(
        baseline * 100,
        color="gray",
        linestyle="--",
        label=f"Overall: {baseline:.2%}",
    )

    ax.set_title(feature)
    ax.set_xlabel("Median parameter value within bin")
    ax.set_ylabel("Defect rate (%)")
    ax.legend()

for ax in axes[len(features):]:
    ax.set_visible(False)

fig.suptitle("Defect Rate Across Parameter Ranges", fontsize=18)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("02_parameter_defect_rates.png", bbox_inches="tight")
plt.show()

bin_results = pd.concat(bin_tables, ignore_index=True)
display(bin_results.round(4))
bin_results.to_csv("parameter_bin_statistics.csv", index=False)

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(24, 7),
    gridspec_kw={"width_ratios": [1.5, 1, 1.3]},
)

# 1. Spearman correlation
corr = df[features + ["defect_flag"]].corr(method="spearman")

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    annot_kws={"size": 9},
    ax=axes[0],
)
axes[0].set_title("Spearman Correlation")

# 2. Defect rate by tool
tool_summary = (
    df.groupby("tool_id")
    .agg(
        sample_count=("defect_flag", "size"),
        defect_rate=("defect_flag", "mean"),
    )
    .sort_values("defect_rate", ascending=False)
)

bars = axes[1].bar(
    tool_summary.index.astype(str),
    tool_summary["defect_rate"] * 100,
    color="#2878B5",
)

axes[1].bar_label(
    bars,
    labels=[
        f"{rate:.2%}\nn={count:,}"
        for rate, count in zip(
            tool_summary["defect_rate"],
            tool_summary["sample_count"],
        )
    ],
    padding=4,
)
axes[1].axhline(baseline * 100, color="gray", linestyle="--")
axes[1].set_title("Defect Rate by Tool")
axes[1].set_xlabel("Tool")
axes[1].set_ylabel("Defect rate (%)")
axes[1].margins(y=0.25)

# 3. Daily and rolling seven-day defect rate
daily = (
    df.set_index("timestamp")
    .resample("D")["defect_flag"]
    .agg(["sum", "count"])
)
daily["defect_rate"] = daily["sum"] / daily["count"].replace(0, np.nan)

rolling_counts = daily[["sum", "count"]].rolling(7, min_periods=7).sum()
daily["rolling_7d"] = (
    rolling_counts["sum"] / rolling_counts["count"].replace(0, np.nan)
)

axes[2].plot(
    daily.index,
    daily["defect_rate"] * 100,
    alpha=0.4,
    label="Daily",
)
axes[2].plot(
    daily.index,
    daily["rolling_7d"] * 100,
    color="#E45756",
    linewidth=2,
    label="7-day pooled rate",
)
axes[2].axhline(baseline * 100, color="gray", linestyle="--")
axes[2].set_title("Defect Rate Over Time")
axes[2].set_xlabel("Date")
axes[2].set_ylabel("Defect rate (%)")
axes[2].tick_params(axis="x", rotation=45)
axes[2].legend()

plt.tight_layout()
plt.savefig("03_correlation_tool_time.png", bbox_inches="tight")
plt.show()

display(tool_summary)

In [ ]:
# 1. 依批次開始時間排序
lot_order = (
    df.groupby("lot_id")["timestamp"]
    .min()
    .sort_values()
    .index
)

# 2. 前 70% 批次訓練，中間 15% 驗證，最後 15% 測試
cut1 = int(len(lot_order) * 0.70)
cut2 = int(len(lot_order) * 0.85)

train = df[df["lot_id"].isin(lot_order[:cut1])].copy()
valid = df[df["lot_id"].isin(lot_order[cut1:cut2])].copy()
test = df[df["lot_id"].isin(lot_order[cut2:])].copy()

# 3. 確認各資料集的時間範圍沒有重疊
assert train["timestamp"].max() < valid["timestamp"].min()
assert valid["timestamp"].max() < test["timestamp"].min()

# 4. 查看筆數、時間範圍與缺陷率
summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(data),
        "lots": data["lot_id"].nunique(),
        "start": data["timestamp"].min(),
        "end": data["timestamp"].max(),
        "defect_rate": data["defect_flag"].mean(),
    }
    for name, data in [
        ("Train", train),
        ("Validation", valid),
        ("Test", test),
    ]
])

display(summary.style.format({"defect_rate": "{:.2%}"}))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
)

# 1. 製程參數作為輸入，缺陷標籤作為答案
X_train = train[features]
y_train = train["defect_flag"]

X_valid = valid[features]
y_valid = valid["defect_flag"]

# 2. 所有前處理規則都由訓練集學習
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000)),
])

# 3. 訓練模型
model.fit(X_train, y_train)

# 4. 預測驗證集的缺陷機率
valid_probability = model.predict_proba(X_valid)[:, 1]

# 暫用 0.5 作為門檻，機率 >= 50% 判為有缺陷
threshold = 0.5
valid_prediction = (valid_probability >= threshold).astype(int)

# 5. 評估結果
metrics = pd.DataFrame({
    "Metric": [
        "Validation defect rate",
        "Average Precision (AP)",
        "ROC-AUC",
        "Precision at 0.5",
        "Recall at 0.5",
    ],
    "Value": [
        y_valid.mean(),
        average_precision_score(y_valid, valid_probability),
        roc_auc_score(y_valid, valid_probability),
        precision_score(y_valid, valid_prediction, zero_division=0),
        recall_score(y_valid, valid_prediction, zero_division=0),
    ],
})

display(metrics.style.format({"Value": "{:.3f}"}))
print(f"Predicted defects: {valid_prediction.sum():,} / {len(y_valid):,}")

# 6. 畫混淆矩陣與 Precision-Recall 曲線
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_valid,
    valid_prediction,
    labels=[0, 1],
    display_labels=["No defect", "Defect"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title("Validation Confusion Matrix (threshold = 0.5)")
axes[0].grid(False)

PrecisionRecallDisplay.from_predictions(
    y_valid,
    valid_probability,
    ax=axes[1],
    name="Logistic Regression",
)
axes[1].axhline(
    y_valid.mean(),
    color="gray",
    linestyle="--",
    label=f"Prevalence baseline: {y_valid.mean():.2%}",
)
axes[1].set_title("Validation Precision-Recall Curve")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

# 沿用前一格的 y_valid 與 valid_probability
results = []

for threshold in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]:
    prediction = (valid_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_valid, prediction, labels=[0, 1]
    ).ravel()

    results.append({
        "Threshold": threshold,
        "Precision": tp / (tp + fp) if tp + fp else 0,
        "Recall": tp / (tp + fn) if tp + fn else 0,
        "Detected defects": tp,
        "Missed defects": fn,
        "False alarms": fp,
        "Total alerts": tp + fp,
        "Alert rate": (tp + fp) / len(y_valid),
    })

threshold_results = pd.DataFrame(results)

display(threshold_results.style.format({
    "Threshold": "{:.2f}",
    "Precision": "{:.1%}",
    "Recall": "{:.1%}",
    "Alert rate": "{:.1%}",
}))

# 左圖看比例，右圖看漏判與誤報的實際數量
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for metric in ["Precision", "Recall"]:
    axes[0].plot(
        threshold_results["Threshold"],
        threshold_results[metric] * 100,
        marker="o",
        label=metric,
    )

axes[0].set(
    title="Precision and Recall by Threshold",
    xlabel="Decision threshold",
    ylabel="Percentage (%)",
    ylim=(0, 100),
)
axes[0].legend()

for column in ["Missed defects", "False alarms"]:
    axes[1].plot(
        threshold_results["Threshold"],
        threshold_results[column],
        marker="o",
        label=column,
    )

axes[1].set(
    title="Missed Defects and False Alarms",
    xlabel="Decision threshold",
    ylabel="Number of records",
)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    PrecisionRecallDisplay,
)

# 1. 建立梯度提升樹，學習非線性關係及參數間的交互作用
tree_model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_leaf_nodes=15,
    min_samples_leaf=50,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42,
)

tree_model.fit(X_train, y_train)

# 2. 兩個模型都在同一份驗證集上評估
lr_probability = model.predict_proba(X_valid)[:, 1]
tree_probability = tree_model.predict_proba(X_valid)[:, 1]

probabilities = {
    "Logistic Regression": lr_probability,
    "Gradient Boosting": tree_probability,
}

# 3. 假設只能檢查風險最高的 20%，比較找出的缺陷數量
# 20% 是專題示範的檢查預算
inspection_fraction = 0.20
inspection_count = int(np.ceil(len(y_valid) * inspection_fraction))
actual = y_valid.to_numpy()

comparison = []

for name, probability in probabilities.items():
    # 同分時保留原本順序，確保選取結果可重現
    selected = np.argsort(-probability, kind="stable")[:inspection_count]
    detected = int(actual[selected].sum())

    comparison.append({
        "Model": name,
        "AP": average_precision_score(actual, probability),
        "ROC-AUC": roc_auc_score(actual, probability),
        "Inspected": inspection_count,
        "Detected defects": detected,
        "Missed defects": int(actual.sum()) - detected,
        "Precision at top 20%": detected / inspection_count,
        "Recall at top 20%": detected / actual.sum(),
    })

comparison = pd.DataFrame(comparison)

display(comparison.style.format({
    "AP": "{:.3f}",
    "ROC-AUC": "{:.3f}",
    "Precision at top 20%": "{:.1%}",
    "Recall at top 20%": "{:.1%}",
}))

# 4. 比較 PR 曲線及相同檢查量下找出的缺陷數
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, probability in probabilities.items():
    PrecisionRecallDisplay.from_predictions(
        actual,
        probability,
        name=name,
        ax=axes[0],
    )

axes[0].axhline(
    actual.mean(),
    color="gray",
    linestyle="--",
    label=f"Prevalence baseline: {actual.mean():.2%}",
)
axes[0].set_title("Validation Precision-Recall Comparison")
axes[0].legend()

bars = axes[1].bar(
    comparison["Model"],
    comparison["Detected defects"],
    color=["#2878B5", "#E45756"],
)
axes[1].bar_label(bars, padding=4)
axes[1].set_title(f"Defects Found in Top {inspection_count:,} Records")
axes[1].set_ylabel("Detected defects")
axes[1].margins(y=0.15)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
)

# 1. 用訓練好的模型預測測試集
X_test = test[features]
y_test = test["defect_flag"].to_numpy()

test_probability = tree_model.predict_proba(X_test)[:, 1]

# 2. 固定檢查風險最高的 20%，選取過程只使用預測分數
inspection_fraction = 0.20
inspection_count = int(np.ceil(len(y_test) * inspection_fraction))

selected = np.argsort(
    -test_probability, kind="stable"
)[:inspection_count]

test_alert = np.zeros(len(y_test), dtype=int)
test_alert[selected] = 1

# 3. 選取完成後，用真實標籤計算結果
detected = int(y_test[selected].sum())
missed = int(y_test.sum()) - detected
false_alerts = inspection_count - detected

test_result = {
    "Dataset": "Test",
    "Records": len(y_test),
    "Defect rate": y_test.mean(),
    "AP": average_precision_score(y_test, test_probability),
    "ROC-AUC": roc_auc_score(y_test, test_probability),
    "Inspected": inspection_count,
    "Detected defects": detected,
    "Missed defects": missed,
    "Precision": detected / inspection_count,
    "Recall": detected / y_test.sum(),
}

# 4. 以相同規則計算驗證集結果，方便比較
valid_actual = np.asarray(y_valid)
valid_scores = tree_model.predict_proba(X_valid)[:, 1]
valid_count = int(np.ceil(len(valid_actual) * inspection_fraction))

valid_selected = np.argsort(
    -valid_scores, kind="stable"
)[:valid_count]

valid_detected = int(valid_actual[valid_selected].sum())

valid_result = {
    "Dataset": "Validation",
    "Records": len(valid_actual),
    "Defect rate": valid_actual.mean(),
    "AP": average_precision_score(valid_actual, valid_scores),
    "ROC-AUC": roc_auc_score(valid_actual, valid_scores),
    "Inspected": valid_count,
    "Detected defects": valid_detected,
    "Missed defects": int(valid_actual.sum()) - valid_detected,
    "Precision": valid_detected / valid_count,
    "Recall": valid_detected / valid_actual.sum(),
}

final_results = pd.DataFrame([valid_result, test_result])

display(final_results.style.format({
    "Defect rate": "{:.1%}",
    "AP": "{:.3f}",
    "ROC-AUC": "{:.3f}",
    "Precision": "{:.1%}",
    "Recall": "{:.1%}",
}))

# 5. 左圖顯示測試集警報結果，右圖比較缺陷找出比例
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_alert,
    labels=[0, 1],
    display_labels=["No defect", "Defect"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title("Test Set: Top 20% Selected for Inspection")
axes[0].set_xlabel("Inspection alert (0 = not selected, 1 = selected)")
axes[0].grid(False)

bars = axes[1].bar(
    final_results["Dataset"],
    final_results["Recall"] * 100,
    color=["#2878B5", "#E45756"],
)
axes[1].bar_label(bars, fmt="%.1f%%", padding=4)
axes[1].set(
    title="Recall at 20% Inspection Budget",
    ylabel="Defects detected (%)",
    ylim=(0, 100),
)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

# 1. 使用驗證集分析，保留既有模型與測試結果
importance_result = permutation_importance(
    tree_model,
    X_valid,
    y_valid,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=2,
)

# 2. 整理每個欄位被打亂後，AP 的平均下降量
importance_table = pd.DataFrame({
    "Feature": X_valid.columns,
    "AP drop": importance_result.importances_mean,
    "Repeat std": importance_result.importances_std,
}).sort_values("AP drop", ascending=False)

display(importance_table.style.format({
    "AP drop": "{:.4f}",
    "Repeat std": "{:.4f}",
}))

# 3. 畫圖，誤差線代表五次打亂結果的標準差
plot_data = importance_table.sort_values("AP drop")

fig, ax = plt.subplots(figsize=(9, 5))

ax.barh(
    plot_data["Feature"],
    plot_data["AP drop"],
    xerr=plot_data["Repeat std"],
    color="#2878B5",
    capsize=4,
)

ax.axvline(0, color="gray", linestyle="--")
ax.set(
    title="Permutation Importance on Validation Set",
    xlabel="Decrease in Average Precision after shuffling",
    ylabel="Feature",
)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

top_features = [
    "alignment_error",
    "temperature_mean",
    "process_time",
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

PartialDependenceDisplay.from_estimator(
    tree_model,
    X_valid,
    features=top_features,
    kind="average",
    method="brute",
    response_method="predict_proba",
    grid_resolution=40,
    percentiles=(0.05, 0.95),
    ax=axes,
    n_jobs=2,
)

for ax, feature in zip(axes, top_features):
    ax.set_title(feature)
    ax.set_ylabel("Average predicted defect probability")

fig.suptitle("Partial Dependence of Top Features")
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import json
import joblib
import shutil
import sklearn
import platform

output_dir = Path("SoIC_Bonding_results")
output_dir.mkdir(exist_ok=True)

# 1. 保存訓練好的模型
joblib.dump(tree_model, output_dir / "bonding_risk_model.joblib")

# 2. 保存模型比較、測試結果與解釋分析
tables = {
    "model_comparison.csv": comparison,
    "final_evaluation.csv": final_results,
    "permutation_importance.csv": importance_table,
    "threshold_comparison_logistic.csv": threshold_results,
}

for filename, table in tables.items():
    table.to_csv(output_dir / filename, index=False)

# 3. 保存特徵順序、檢查規則與套件版本
metadata = {
    "model": "HistGradientBoostingClassifier",
    "features": list(features),
    "target": "defect_flag",
    "target_definition": {"0": "no defect", "1": "defect"},
    "inspection_policy": {
        "type": "top_fraction_per_batch",
        "fraction": 0.20,
        "tie_break": "original row order",
    },
    "data_source": "synthetic data",
    "feature_units": {feature: None for feature in features},
    "units_status": "not documented; require definition",
    "versions": {
        "python": platform.python_version(),
        "scikit_learn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "joblib": joblib.__version__,
    },
}

(output_dir / "model_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# 4. 產生專題說明草稿，評估數值取自執行結果
readme = f"""# SoIC Bonding Quality Risk Prediction

## Objective
Rank synthetic bonding records by predicted defect risk,
then select the highest-risk 20% for inspection.

## Dataset
100,000 synthetic records with seven process features.
Feature units and process-step assumptions remain undocumented.
These records do not establish actual TSMC SoIC operating conditions.

## Method
Chronological, lot-separated train/validation/test split.
Logistic Regression baseline and HistGradientBoostingClassifier.
Permutation importance and partial dependence for model explanation.

## Evaluation
{final_results.to_csv(index=False)}

## Limitations
The full dataset was explored during EDA before model evaluation.
Results represent a learning project using synthetic data.
The inspection policy requires a complete batch for ranking.
Predicted probabilities have not been assessed for calibration.
Model explanations describe associations, not process causality.

## Saved model
Load bonding_risk_model.joblib with joblib,
and supply numeric features in the order in model_metadata.json.
Use only trusted model files.
"""

(output_dir / "README.md").write_text(readme, encoding="utf-8")

# 5. 壓縮並下載
zip_path = shutil.make_archive(
    "SoIC_Bonding_results",
    "zip",
    root_dir=output_dir,
)

print("Saved files:")
for path in sorted(output_dir.iterdir()):
    print(f"  {path.name}")

from google.colab import files
files.download(zip_path)

In [ ]:
from pathlib import Path
from sklearn.metrics import PrecisionRecallDisplay
from sklearn.inspection import PartialDependenceDisplay
import shutil

image_dir = Path("SoIC_Bonding_figures")
image_dir.mkdir(exist_ok=True)

# 1. 兩個模型的 Precision-Recall 曲線
fig, ax = plt.subplots(figsize=(8, 6))

for name, probability in {
    "Logistic Regression": lr_probability,
    "Gradient Boosting": tree_probability,
}.items():
    PrecisionRecallDisplay.from_predictions(
        y_valid, probability, name=name, ax=ax
    )

ax.axhline(
    np.asarray(y_valid).mean(),
    color="gray",
    linestyle="--",
    label="Prevalence baseline",
)
ax.set_title("Validation Precision-Recall Comparison")
ax.legend()
fig.tight_layout()
fig.savefig(image_dir / "01_model_comparison.png", dpi=200)
plt.close(fig)

# 2. 驗證集與測試集的缺陷找出比例
fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    final_results["Dataset"],
    final_results["Recall"] * 100,
    color=["#2878B5", "#E45756"],
)
ax.bar_label(bars, fmt="%.1f%%", padding=4)
ax.set(
    title="Recall at 20% Inspection Budget",
    ylabel="Defects detected (%)",
    ylim=(0, 100),
)
fig.tight_layout()
fig.savefig(image_dir / "02_final_evaluation.png", dpi=200)
plt.close(fig)

# 3. 特徵重要性
plot_data = importance_table.sort_values("AP drop")
fig, ax = plt.subplots(figsize=(9, 5))

ax.barh(
    plot_data["Feature"],
    plot_data["AP drop"],
    xerr=plot_data["Repeat std"],
    color="#2878B5",
    capsize=4,
)
ax.axvline(0, color="gray", linestyle="--")
ax.set(
    title="Permutation Importance on Validation Set",
    xlabel="Decrease in Average Precision after shuffling",
)
fig.tight_layout()
fig.savefig(image_dir / "03_feature_importance.png", dpi=200)
plt.close(fig)

# 4. 主要參數的部分依賴圖
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

PartialDependenceDisplay.from_estimator(
    tree_model,
    X_valid,
    features=[
        "alignment_error",
        "temperature_mean",
        "process_time",
    ],
    kind="average",
    method="brute",
    response_method="predict_proba",
    grid_resolution=40,
    percentiles=(0.05, 0.95),
    ax=axes,
    n_jobs=2,
)

for ax in axes:
    ax.set_ylabel("Average predicted defect probability")

fig.suptitle("Partial Dependence of Top Features")
fig.tight_layout()
fig.savefig(image_dir / "04_partial_dependence.png", dpi=200)
plt.close(fig)

# 5. 壓縮並下載四張成果圖
zip_path = shutil.make_archive(
    "SoIC_Bonding_figures",
    "zip",
    root_dir=image_dir,
)

from google.colab import files
files.download(zip_path)